# Lesson 8: Instruction tuning and answer masking

Turn question–answer examples into training tensors and compute loss only on the answer portion.

**How to run:** Select a Python kernel with PyTorch installed, then run each code cell from top to bottom with **Shift+Enter**. This notebook is self-contained; no other notebook needs to run first. Restart the kernel and run from the top to reset the experiment.

**Source:** This lesson was developed from the [reference conversation's roadmap](https://chatgpt.com/share/6aa56bca-4a1c-83e9-9153-1edcc7ff7e40). The reference supplies Lesson 1 and a topic outline; Lessons 2–12 are newly written implementations of those topics. Small examples demonstrate the mechanics; they are not trained assistants.


In [ ]:
import math
from pathlib import Path
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
# Small tensors can be slower with many CPU threads.
torch.set_num_threads(1)
device = torch.device('cpu')
print('PyTorch:', torch.__version__, '| device:', device)


## Format instruction–response examples

These synthetic examples demonstrate supervised fine-tuning (SFT). The model sees a prompt followed by an answer. We mask prompt targets with `-100`, PyTorch's default ignored target, so only answer tokens contribute to loss. A real SFT workflow starts from a meaningfully pretrained model and uses diverse, reviewed examples.


In [ ]:
examples = [
    ('What stores objects?', 'S3 stores objects.'),
    ('What runs virtual machines?', 'EC2 runs virtual machines.'),
    ('What manages permissions?', 'IAM manages permissions.'),
    ('What resolves domain names?', 'DNS resolves domain names.'),
]
held_out = ('Which service stores objects?', 'S3 stores objects.')

def format_prompt(question):
    return 'Question: ' + question + '\nAnswer: '

# ASCII base vocabulary avoids depending on characters in held-out examples.
chars = [chr(i) for i in range(128)]
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
def encode(s): return [stoi[ch] for ch in s]
def decode(ids): return ''.join(itos[int(i)] for i in ids)
block_size = 96

def make_sft_batch(pairs):
    inputs, labels = [], []
    for question, answer in pairs:
        prompt = format_prompt(question)
        full = encode(prompt + answer + '\n')
        if len(full) - 1 > block_size:
            raise ValueError('Example is too long for the context window.')
        x, y = full[:-1], full[1:]
        # Target index len(prompt)-1 predicts the first answer character.
        y[:len(prompt) - 1] = [-100] * (len(prompt) - 1)
        padding = block_size - len(x)
        inputs.append(x + [0] * padding)
        labels.append(y + [-100] * padding)
    return torch.tensor(inputs, device=device), torch.tensor(labels, device=device)

sft_x, sft_y = make_sft_batch(examples)
print('Batch:', sft_x.shape, '| supervised answer tokens:', (sft_y != -100).sum().item())
first_answer_index = len(format_prompt(examples[0][0])) - 1
assert (sft_y[0, :first_answer_index] == -100).all()
assert sft_y[0, first_answer_index].item() == encode(examples[0][1])[0]


## Causal multi-head attention

Each head compares queries to keys, then combines value vectors. A triangular mask prevents reading future tokens. This is the implementation developed in Lessons 3–4, included here so this notebook runs independently.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, width, heads, context_length, dropout=0.0):
        super().__init__()
        assert width % heads == 0
        self.heads = heads
        self.head_size = width // heads
        self.qkv = nn.Linear(width, 3 * width, bias=False)
        self.projection = nn.Linear(width, width)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('causal_mask', torch.tril(torch.ones(context_length, context_length, dtype=torch.bool)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # Give each head its own vector slice: [B, heads, T, head_size].
        q = q.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        k = k.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        v = v.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_size)
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float('-inf'))
        weights = self.dropout(F.softmax(scores, dim=-1))
        out = (weights @ v).transpose(1, 2).contiguous().reshape(B, T, C)
        return self.projection(out)


## Transformer block

LayerNorm normalizes each token's features. Residual additions let information pass around attention and the MLP. The MLP expands each token vector, applies a nonlinear function, and projects it back.


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, width, heads, context_length, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(width)
        self.attention = CausalSelfAttention(width, heads, context_length, dropout)
        self.ln2 = nn.LayerNorm(width)
        self.mlp = nn.Sequential(
            nn.Linear(width, 4 * width), nn.GELU(),
            nn.Linear(4 * width, width), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


## Token and position embeddings → vocabulary scores

Token embeddings describe characters; learned position embeddings distinguish their positions. The final linear layer predicts the next character at every position. Cross entropy consumes raw scores (logits).


In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, context_length, width=32, heads=4, layers=2):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, width)
        self.position_embedding = nn.Embedding(context_length, width)
        self.blocks = nn.Sequential(*[
            TransformerBlock(width, heads, context_length) for _ in range(layers)
        ])
        self.final_norm = nn.LayerNorm(width)
        self.lm_head = nn.Linear(width, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.context_length:
            raise ValueError('Sequence exceeds context length.')
        positions = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        logits = self.lm_head(self.final_norm(self.blocks(x)))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

model = TinyGPT(vocab_size, block_size).to(device)
print('Parameters:', sum(p.numel() for p in model.parameters()))


## A short supervised training run

This notebook starts from random weights to keep the mechanics self-contained. It therefore demonstrates the SFT objective rather than the capabilities of a pretrained instruction model. The four examples are far too small to support claims of general question answering.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
model.train()
for step in range(100):
    _, loss = model(sft_x, sft_y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 25 == 0:
        print(step, round(loss.item(), 4))


## Compare a memorized prompt with an unseen wording


In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=80, temperature=1.0, top_k=None, top_p=None):
    if not prompt:
        raise ValueError('Provide a nonempty prompt using characters from input.txt.')
    if temperature < 0:
        raise ValueError('temperature must be nonnegative; 0 selects greedily.')
    if top_k is not None and top_k < 1:
        raise ValueError('top_k must be at least 1.')
    if top_p is not None and not 0 < top_p <= 1:
        raise ValueError('top_p must be in (0, 1].')
    target_device = next(model.parameters()).device
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=target_device)
    was_training = model.training
    model.eval()
    try:
        for _ in range(max_new_tokens):
            logits, _ = model(idx[:, -model.context_length:])
            logits = logits[:, -1, :]
            if temperature == 0:
                next_id = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k is not None:
                    threshold = torch.topk(logits, min(top_k, logits.size(-1))).values[:, -1:]
                    logits = logits.masked_fill(logits < threshold, float('-inf'))
                if top_p is not None:
                    sorted_logits, sorted_ids = logits.sort(descending=True)
                    cumulative = sorted_logits.softmax(dim=-1).cumsum(dim=-1)
                    # Keep the first token that takes cumulative mass over p.
                    remove = cumulative > top_p
                    remove[:, 1:] = remove[:, :-1].clone()
                    remove[:, 0] = False
                    sorted_logits = sorted_logits.masked_fill(remove, float('-inf'))
                    logits = torch.full_like(logits, float('-inf')).scatter(1, sorted_ids, sorted_logits)
                next_id = torch.multinomial(logits.softmax(dim=-1), num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return decode(idx[0].tolist())
    finally:
        model.train(was_training)


In [ ]:
for question in (examples[0][0], held_out[0]):
    print(repr(generate(model, format_prompt(question), max_new_tokens=25, temperature=0)))
model.eval()
with torch.no_grad():
    _, train_loss = model(sft_x, sft_y)
    held_x, held_y = make_sft_batch([held_out])
    _, held_loss = model(held_x, held_y)
print('Training answer loss:', train_loss.item())
print('Held-out answer loss:', held_loss.item())


## Try it yourself

Print the non-ignored target characters for one example and confirm they contain only the answer plus newline. Add varied training questions while keeping a separate test set. Compare the held-out wording against the training wording.
